## Step 0: Install Required Packages
Run this once. Restart the kernel afterwards if any of these were just installed for the first time.

In [14]:
# !pip install streamlit easyocr deep-translator gtts pillow numpy -q

## Part A: OCR Text Extraction (test in-notebook)
Step 1: Import Required Libraries

In [15]:
import easyocr
from PIL import Image
import numpy as np

Step 2: Load the EasyOCR Reader

In [16]:
# 'en' for English; add more language codes if your documents mix scripts
reader = easyocr.Reader(['en'])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Step 3: Load an Image

**IMPORTANT:** Upload an actual image file into the same folder as this notebook first (e.g. via the Jupyter file browser), then update the filename below to match it exactly. This will throw `FileNotFoundError` until a real file is in place.

In [17]:
image = Image.open("images.jpg").convert('RGB')  # <-- replace with your actual uploaded filename
image_np = np.array(image)

Step 4: Run OCR

In [18]:
results = reader.readtext(image_np)

c:\Users\adhik\miniconda3\envs\condaTetron\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step 5: Extract and Print the Text

In [19]:
extracted_text = " ".join([res[1] for res in results])
print("Extracted Text:", extracted_text)

Extracted Text: 66 It always seems impossible until it's done_ Nelson Mandela


## Part B: Text Translation (test in-notebook)

In [20]:
# Part B — reuse Part A's actual output
from deep_translator import GoogleTranslator

translated = GoogleTranslator(source='auto', target='ne').translate(extracted_text)
print(translated)

66 यो सधैं असम्भव देखिन्छ जब सम्म यो पूरा हुँदैन_ नेल्सन मन्डेला


## Part C: Text-to-Speech (test in-notebook)

In [21]:
# Part C — reuse Part B's actual output
from gtts import gTTS
from IPython.display import Audio

tts = gTTS(text=translated, lang='ne')
tts.save("translated_speech.mp3")
Audio("translated_speech.mp3", autoplay=True)

## Part D: Combined Pipeline as a Streamlit App

**Note:** This cell does NOT run the dashboard directly. Streamlit apps cannot execute inside a normal Jupyter cell — `st.file_uploader`, `st.button`, etc. only work when the code is run by the `streamlit run` command as a standalone script. This `%%writefile` magic writes the code below to `app.py` on disk. The next cell then launches it.

In [22]:
%%writefile app.py
import streamlit as st
import easyocr
import numpy as np
from PIL import Image
from gtts import gTTS
from deep_translator import GoogleTranslator
from io import BytesIO

st.set_page_config(page_title="Multilingual Document Reader", layout="centered")

# ---------------------------
# Load OCR reader (cached so it doesn't reload on every rerun)
# ---------------------------
@st.cache_resource
def load_reader():
    return easyocr.Reader(['en'])  # add more codes e.g. ['en', 'hi'] if needed

reader = load_reader()

# ---------------------------
# Language options for translation / speech
# ---------------------------
LANGUAGES = {
    "Nepali": "ne",
    "Hindi": "hi",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Japanese": "ja",
}

st.title("\U0001F4C4 Multilingual Document Reader")
st.write("Upload a photo of a document, sign, or menu — get it read aloud in your chosen language.")

uploaded_file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])
camera_file = st.camera_input("Or capture from camera")

image_file = uploaded_file or camera_file

if image_file:
    image = Image.open(image_file).convert("RGB")
    st.image(image, caption="Selected Image", use_container_width=True)

    # ---------------------------
    # Stage 1: OCR (Vision)
    # ---------------------------
    st.subheader("1\uFE0F\u20E3 Extracted Text")
    with st.spinner("Reading text from image..."):
        image_np = np.array(image)
        results = reader.readtext(image_np)
        extracted_text = " ".join([res[1] for res in results]).strip()

    if not extracted_text:
        st.warning("No text detected — try a clearer or closer image.")
    else:
        st.text_area("OCR Output (editable before translating)", extracted_text, height=120, key="ocr_text")

        # ---------------------------
        # Stage 2: Translation (NLP)
        # ---------------------------
        st.subheader("2\uFE0F\u20E3 Translate")
        target_lang_name = st.selectbox("Translate to:", list(LANGUAGES.keys()))
        target_lang_code = LANGUAGES[target_lang_name]

        text_to_translate = st.session_state.get("ocr_text", extracted_text)

        if st.button("Translate & Read Aloud"):
            with st.spinner("Translating..."):
                translated = GoogleTranslator(source='auto', target=target_lang_code).translate(text_to_translate)
            st.write(f"**{target_lang_name}:**", translated)

            # ---------------------------
            # Stage 3: Text-to-Speech (Speech)
            # ---------------------------
            st.subheader("3\uFE0F\u20E3 Listen")
            with st.spinner("Generating audio..."):
                tts = gTTS(text=translated, lang=target_lang_code)
                mp3_bytes = BytesIO()
                tts.write_to_fp(mp3_bytes)
                mp3_bytes.seek(0)
            st.audio(mp3_bytes.read(), format="audio/mp3")

Overwriting app.py
